# Solución de la Ecuación de Difusión — Método de Crank-Nicolson

**Problema:** Distribución de temperatura en una pared infinita de ancho $l = 1$ m

**Ecuación de difusión:**
$$\frac{\partial T}{\partial t} = \alpha^2 \frac{\partial^2 T}{\partial x^2}, \quad x \in (0, l),\; t > 0$$

**Condiciones de frontera:**
$$T(0,t) = 0 \qquad \frac{\partial T}{\partial x}\bigg|_{x=l} = 0$$

**Condición inicial:**
$$T(x, 0) = 100 \quad (^\circ C)$$

---

## Fundamento del Método de Crank-Nicolson

El método de Crank-Nicolson es un esquema de diferencias finitas **implícito de segundo orden** tanto en espacio como en tiempo. Promedia la discretización espacial entre el nivel temporal $n$ (explícito) y el nivel $n+1$ (implícito):

$$\frac{T_i^{n+1} - T_i^n}{\Delta t} = \frac{\alpha^2}{2}\left[\frac{T_{i+1}^n - 2T_i^n + T_{i-1}^n}{(\Delta x)^2} + \frac{T_{i+1}^{n+1} - 2T_i^{n+1} + T_{i-1}^{n+1}}{(\Delta x)^2}\right]$$

Definiendo $r = \dfrac{\alpha^2 \Delta t}{(\Delta x)^2}$, la ecuación se reorganiza como:

$$-\frac{r}{2}T_{i-1}^{n+1} + (1+r)T_i^{n+1} - \frac{r}{2}T_{i+1}^{n+1} = \frac{r}{2}T_{i-1}^n + (1-r)T_i^n + \frac{r}{2}T_{i+1}^n$$

Esto produce un **sistema tridiagonal** $A\,\mathbf{T}^{n+1} = B\,\mathbf{T}^n$ que se resuelve en cada paso de tiempo.

**Ventaja clave:** El método es **incondicionalmente estable** para cualquier valor de $r$, a diferencia de los esquemas explícitos que requieren $r \leq 0.5$.

## Parámetros del problema

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.linalg import solve_banded
import pandas as pd
import time

# ── Parámetros del problema (idénticos a MELIN) ──────────────
alpha  = 0.2        # Constante de difusión térmica [m²/s^(1/2)]
L      = 1.0        # Ancho de la pared [m]
N      = 10         # Número de divisiones espaciales
dx     = L / N      # Paso espacial
T0     = 100.0      # Temperatura inicial [°C]

# ── Parámetros temporales ────────────────────────────────────
dt     = 0.05       # Paso de tiempo [s]  (mismo que MELIN h=0.05)
t_end  = 30.0       # Tiempo final [s]
NPT    = int(t_end / dt)   # Número de pasos de tiempo

# ── Número de Fourier (parámetro de estabilidad) ─────────────
r = alpha**2 * dt / dx**2

# ── Grid espacial ────────────────────────────────────────────
x = np.linspace(0, L, N+1)   # N+1 puntos: x_0, x_1, ..., x_N

print(f"α     = {alpha}")
print(f"l     = {L} m")
print(f"N     = {N}  nodos")
print(f"Δx    = {dx}")
print(f"Δt    = {dt}")
print(f"NPT   = {NPT} pasos de tiempo")
print(f"r     = α²Δt/Δx² = {r:.4f}")
print()
print("Crank-Nicolson es incondicionalmente estable → cualquier r es válido ✔")
print(f"(r = {r:.4f}, equivalente al MELIN: r = {alpha**2 * 0.05 / dx**2:.4f})")

## Construcción de las matrices del sistema tridiagonal

El sistema $A\,\mathbf{T}^{n+1} = B\,\mathbf{T}^n$ involucra dos matrices tridiagonales:

$$A = \begin{pmatrix} 1 & 0 & & \\ -r/2 & 1+r & -r/2 & \\ & \ddots & \ddots & \ddots \\ & & -r/2 & 1+r & -r/2 \\ & & & a_{N,N-1} & a_{N,N} \end{pmatrix}$$

La frontera Neumann en $x=l$ ($\partial T/\partial x|_{x=l}=0$) se incorpora usando la misma aproximación de orden $\mathcal{O}(\Delta x^2)$ del MELIN:

$$T_{N+1} = \frac{4T_N - T_{N-1}}{3} \implies \text{fila } N: \quad -\frac{r}{3}T_{N-1}^{n+1} + \left(1 + \frac{2r}{3}\right)T_N^{n+1} = \cdots$$

In [ ]:
def construir_matrices(N, r):
    """
    Construye las matrices A (implícita) y B (explícita) del sistema
    tridiagonal de Crank-Nicolson para la ecuación de difusión 1D.

    Frontera x=0  : Dirichlet  T=0  → fila 0 de identidad
    Nodos int.    : i=1,...,N-1 → esquema CN estándar
    Frontera x=l  : Neumann ∂T/∂x=0 usando T_{N+1}=(4T_N-T_{N-1})/3
    """
    size = N + 1   # índices 0..N

    A = np.zeros((size, size))
    B = np.zeros((size, size))

    # ── Fila 0: Dirichlet T(0,t)=0 ──────────────────────────
    A[0, 0] = 1.0
    B[0, 0] = 0.0          # RHS siempre 0 → T_0^{n+1} = 0

    # ── Filas interiores i = 1 … N-1 ────────────────────────
    for i in range(1, N):
        A[i, i-1] = -r/2
        A[i, i  ] =  1 + r
        A[i, i+1] = -r/2

        B[i, i-1] =  r/2
        B[i, i  ] =  1 - r
        B[i, i+1] =  r/2

    # ── Fila N: Neumann (∂T/∂x=0 en x=l) ───────────────────
    # Sustituyendo T_{N+1} = (4T_N - T_{N-1})/3 en el esquema CN:
    # Lado implícito:
    #   -r/2 * T_{N-1}^{n+1}  +  (1+r)*T_N^{n+1}  -  r/2 * (4T_N-T_{N-1})/3
    # = -r/2*(1-1/3)*T_{N-1}^{n+1}  + (1+r - 2r/3)*T_N^{n+1}
    # = -r/3 * T_{N-1}^{n+1}  +  (1 + r/3)*T_N^{n+1}
    A[N, N-1] = -r/3
    A[N, N  ] =  1 + r/3

    # Lado explícito (análogo):
    B[N, N-1] =  r/3
    B[N, N  ] =  1 - r/3

    return A, B


A, B = construir_matrices(N, r)

print("Matriz A (implícita) — primeras 5×5:")
print(np.round(A[:5, :5], 4))
print()
print("Matriz B (explícita) — primeras 5×5:")
print(np.round(B[:5, :5], 4))

## Solución analítica (referencia)

$$T(x,t) = \frac{400}{\pi} \sum_{n=0}^{\infty} \frac{1}{2n+1} \sin\!\left(\frac{2n+1}{2l}\pi x\right) \exp\!\left(-\alpha^2\left(\frac{2n+1}{2l}\pi\right)^2 t\right)$$

In [ ]:
def T_analitica(x, t, alpha=alpha, L=L, n_terms=50):
    """Solución analítica por separación de variables (N términos)."""
    T = np.zeros_like(x, dtype=float)
    for n in range(n_terms):
        k  = (2*n + 1) / (2*L) * np.pi
        T += (1/(2*n+1)) * np.sin(k*x) * np.exp(-alpha**2 * k**2 * t)
    return (400/np.pi) * T

## Integración temporal — Bucle Crank-Nicolson

En cada paso de tiempo se resuelve el sistema lineal:
$$A \cdot \mathbf{T}^{n+1} = B \cdot \mathbf{T}^n$$
usando el solucionador de álgebra lineal de NumPy (`np.linalg.solve`), que internamente aplica eliminación gaussiana con pivoteo parcial.

In [ ]:
# ── Condición inicial ────────────────────────────────────────
T_cn        = np.full(N+1, T0)   # T_i(t=0) = 100 para i=1..N
T_cn[0]     = 0.0                 # Dirichlet T(0,0)=0

# ── Almacenamiento de la solución ───────────────────────────
t_vals      = np.arange(0, t_end + dt, dt)   # tiempos de salida
T_hist      = np.zeros((N+1, len(t_vals)))    # T[nodo, tiempo]
T_hist[:, 0] = T_cn.copy()

# ── Factorización LU de A (solo una vez, fuera del bucle) ───
# np.linalg.solve la hace internamente; para mayor eficiencia
# en sistemas grandes conviene usar scipy.linalg.lu_factor/solve
from scipy.linalg import lu_factor, lu_solve
lu, piv = lu_factor(A)

# ── Cronómetro ───────────────────────────────────────────────
t_inicio_cn = time.perf_counter()

# ── Bucle temporal ───────────────────────────────────────────
T_actual = T_cn.copy()
for k in range(1, len(t_vals)):
    rhs           = B @ T_actual       # lado derecho
    rhs[0]        = 0.0               # Dirichlet: T_0 siempre 0
    T_nuevo       = lu_solve((lu, piv), rhs)
    T_nuevo[0]    = 0.0               # reforzar Dirichlet
    T_hist[:, k]  = T_nuevo
    T_actual      = T_nuevo

t_fin_cn   = time.perf_counter()
t_ejec_cn  = t_fin_cn - t_inicio_cn

# ── Temperatura en la frontera aislada (x=l) ────────────────
T_frontera_cn = (4*T_hist[N, :] - T_hist[N-1, :]) / 3.0

print(f"Integración completada: {len(t_vals)} pasos")
print(f"=========================================")
print(f"  Tiempo de ejecución : {t_ejec_cn:.6f} segundos")
print(f"  Archivo generado    : CN.dat")
print(f"=========================================")

## Guardado del archivo CN.dat

El formato es idéntico al `MELINP.dat` generado por el método de líneas:
`tiempo | T_1 | T_2 | ... | T_N | T_frontera`

In [ ]:
# ── CN.dat: columnas = [t, T_1..T_N, T_{N+1}] ───────────────
datos_cn = np.column_stack([
    t_vals,
    T_hist[:N, :].T,       # nodos 1..N (sin el nodo 0 que es siempre 0)
    T_frontera_cn          # temperatura virtual en x=l
])

# Encabezado con nombres de columnas
header = "t" + "".join([f"      T{i+1}" for i in range(N)]) + "  T_frontera"
np.savetxt('CN.dat', datos_cn, fmt='%10.4f', header=header, comments='')

print("Primeras 5 filas de CN.dat:")
df_preview = pd.DataFrame(
    datos_cn[:5],
    columns=["t"] + [f"T{i+1}" for i in range(N)] + ["T_frontera"]
)
display(df_preview.round(4))

## Gráfica 1 — Superficie 3D $T(x,t)$

In [ ]:
# Submuestreo para la superficie (cada 10 pasos de tiempo)
step_t = 10
TT, XX = np.meshgrid(t_vals[::step_t], x)
ZZ     = T_hist[:, ::step_t]

fig = plt.figure(figsize=(10, 6))
ax  = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(TT, XX, ZZ, cmap=cm.plasma, alpha=0.9,
                        linewidth=0, antialiased=True)
fig.colorbar(surf, ax=ax, shrink=0.45, label='Temperatura [°C]')
ax.set_xlabel('Tiempo [s]',    fontsize=11)
ax.set_ylabel('Posición x [m]', fontsize=11)
ax.set_zlabel('T [°C]',         fontsize=11)
ax.set_title('$T(x,t)$ — Crank-Nicolson', fontsize=13)
plt.tight_layout()
plt.show()

## Gráfica 2 — Perfiles de temperatura $T(x)$ en tiempos seleccionados

In [ ]:
tiempos_plot = [0.05, 1.0, 5.0, 10.0, 20.0, 30.0]
colores      = plt.cm.viridis(np.linspace(0.1, 0.9, len(tiempos_plot)))
x_fino       = np.linspace(0, L, 300)

fig, ax = plt.subplots(figsize=(9, 5))

for t_plot, color in zip(tiempos_plot, colores):
    idx      = np.argmin(np.abs(t_vals - t_plot))
    T_cn_plot = T_hist[:, idx]
    T_an_plot = T_analitica(x_fino, t_vals[idx])

    ax.plot(x_fino, T_an_plot,  '--', color=color, alpha=0.7, linewidth=1.2)
    ax.plot(x,      T_cn_plot,  'o-', color=color, linewidth=1.8,
            label=f't = {t_vals[idx]:.2f} s')

ax.set_xlabel('Posición x [m]', fontsize=12)
ax.set_ylabel('Temperatura [°C]', fontsize=12)
ax.set_title('Perfiles de temperatura — Crank-Nicolson (marcadores) vs Analítica (línea discontinua)',
             fontsize=11)
ax.legend(fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Gráfica 3 — Evolución temporal en nodos seleccionados

In [ ]:
nodos_plot  = [1, 3, 5, 7, 9, 10]   # índices de nodos a graficar
colores_n   = plt.cm.tab10(np.linspace(0, 1, len(nodos_plot)))

fig, ax = plt.subplots(figsize=(9, 5))

for nodo, color in zip(nodos_plot, colores_n):
    xi        = x[nodo]
    T_an_t    = np.array([T_analitica(np.array([xi]), t)[0] for t in t_vals])

    ax.plot(t_vals, T_hist[nodo, :], '-',  color=color, linewidth=1.8,
            label=f'CN  x={xi:.1f} m')
    ax.plot(t_vals, T_an_t,          '--', color=color, linewidth=1.0, alpha=0.7)

ax.set_xlabel('Tiempo [s]', fontsize=12)
ax.set_ylabel('Temperatura [°C]', fontsize=12)
ax.set_title('Evolución temporal T(x_i, t) — Crank-Nicolson (sólido) vs Analítica (discontinuo)',
             fontsize=11)
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Gráfica 4 — Error absoluto respecto a la solución analítica

In [ ]:
# Error absoluto en cada nodo y tiempo
T_analitica_grid = np.zeros_like(T_hist)
for k, t in enumerate(t_vals):
    T_analitica_grid[:, k] = T_analitica(x, t)

error_abs = np.abs(T_hist - T_analitica_grid)

# MAE promedio en el tiempo para cada nodo
mae_nodo = np.mean(error_abs, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Mapa de calor del error
step_e   = 5
TT_e, XX_e = np.meshgrid(t_vals[::step_e], x)
im = axes[0].contourf(TT_e, XX_e, error_abs[:, ::step_e],
                       levels=30, cmap='hot_r')
fig.colorbar(im, ax=axes[0], label='|Error| [°C]')
axes[0].set_xlabel('Tiempo [s]', fontsize=11)
axes[0].set_ylabel('Posición x [m]', fontsize=11)
axes[0].set_title('Error absoluto $|T_{CN} - T_{\\rm analítica}|$', fontsize=11)

# MAE por nodo
axes[1].bar(x, mae_nodo, width=dx*0.7, color='steelblue',
            edgecolor='gray', linewidth=0.5)
axes[1].set_xlabel('Posición x [m]', fontsize=11)
axes[1].set_ylabel('MAE promedio [°C]', fontsize=11)
axes[1].set_title('Error medio absoluto por nodo', fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Análisis de error — Crank-Nicolson vs Solución analítica', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nError MAE global (promedio sobre todos los nodos y tiempos): {np.mean(error_abs):.6f} °C")

## Tabla comparativa en tiempos representativos

Comparación entre la solución de Crank-Nicolson y la solución analítica en $t = 0.05$, $t = 5.0$, y $t = 30.0$ s.

In [ ]:
for t_comp in [0.05, 5.0, 30.0]:
    idx    = np.argmin(np.abs(t_vals - t_comp))
    T_an   = T_analitica(x, t_vals[idx])
    T_cn_v = T_hist[:, idx]
    err    = np.abs(T_cn_v - T_an)

    df = pd.DataFrame({
        'x [m]'         : np.round(x, 2),
        'CN [°C]'       : np.round(T_cn_v, 4),
        'Analítica [°C]': np.round(T_an,   4),
        '|Error| [°C]'  : np.round(err,    6),
    })
    print(f"\n{'='*55}")
    print(f"  t = {t_vals[idx]:.4f} s")
    print(f"{'='*55}")
    display(df)

## Resumen de métricas del método

In [ ]:
mae_global = np.mean(error_abs)
rmse_global = np.sqrt(np.mean((T_hist - T_analitica_grid)**2))
max_error   = np.max(error_abs)

print("╔══════════════════════════════════════════════╗")
print("║     RESUMEN — MÉTODO DE CRANK-NICOLSON       ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  α (difusividad)       = {alpha}              ║")
print(f"║  N (nodos espaciales)  = {N}               ║")
print(f"║  Δx                    = {dx}             ║")
print(f"║  Δt                    = {dt}            ║")
print(f"║  r = α²Δt/Δx²          = {r:.4f}          ║")
print(f"║  Pasos de tiempo       = {NPT}             ║")
print(f"║  Estabilidad           = incondicional     ║")
print("╠══════════════════════════════════════════════╣")
print(f"║  Tiempo de ejecución   = {t_ejec_cn:.6f} s    ║")
print(f"║  MAE global            = {mae_global:.6f} °C  ║")
print(f"║  RMSE global           = {rmse_global:.6f} °C  ║")
print(f"║  Error máximo          = {max_error:.6f} °C  ║")
print("╚══════════════════════════════════════════════╝")